# Проект 1. Почему замедляется рост подписной базы ПРАЙМ?

**Бизнес-вопрос:** как меняется чистый прирост активных подписок и чем объясняется его замедление: притоком оформлений, размером базы или частотой завершений?

**Исходная гипотеза:** при почти неизменном числе новых оформлений растущая база дает больше завершений, поэтому чистый прирост уменьшается, ухудшение наблюдаемой доли удержания для этого не обязательно.

**Результат:** в феврале 2025-июле 2026 база на конец месяца выросла с 107736 до 192341 подписок. Прирост снизился с 233 до 127 подписок в день, приток находился в диапазоне примерно 354–370 оформлений в день. Доля завершений стартовой базы, приведенная к 30 дням, снизилась с 4,06% до 3,80%. Основное замедление связано с большим числом завершений из выросшей базы

**Воспроизводимость исследования.** Для оформления графиков дополнительно установлена библиотека Seaborn. Зависимости проекта обновлены в `pyproject.toml` и `uv.lock`, поэтому их можно установить командой `uv sync`.

## Как выбиралась тема

Перед выбором темы мы рассмотрели несколько гипотез о поведении пользователей и удержании подписок. Для предварительной проверки сравнили показатели разных групп клиентов в нескольких временных периодах.

| Гипотеза | Как проверяли | Что получили |
|---|---|---|
| Приглашение близких связано с лучшим удержанием | Сравнили долю завершений подписок у клиентов с разным количеством приглашенных | Различия небольшие и неустойчивые между периодами |
| Регулярные покупки связаны с лучшим удержанием | Сравнили долю завершений у клиентов с разной частотой покупок | Устойчивой связи не обнаружили |
| Покупки в первую неделю связаны с последующим оформлением подписки | Сравнили частоту последующего оформления у клиентов с покупками и без них | Результаты различались между периодами, устойчивого преимущества не выявили |

Предварительные проверки не показали устойчивых закономерностей, поэтому мы решили сосредоточиться на динамике всей подписной базы.

Выбранная тема позволяет исследовать конкретный бизнес-вопрос: почему замедляется прирост подписок - из-за снижения притока новых оформлений или увеличения числа завершений? Для ответа можно использовать данные за длительный период, сравнить основные показатели и оценить вклад отдельных факторов в изменение прироста

## Данные, период, уровень детализации и сегменты

**1. Источник данных**

Для исследования используем таблицу `prime.subscriptions`, содержащую информацию об оформлении и завершении подписок. Всего в исходной таблице 333 905 записей. Анализируем всю подписную базу без ограничений по клиентам, тарифам или партнерам, так как наша задача - изучить общий прирост подписок, а не отдельные категории пользователей

**2. Уровень детализации данных (зерно - что представляет собой одна строка таблицы)**

- **В исходной таблице** одна строка соответствует одному оформлению подписки (`subscription_id`).
- **В нашей итоговой таблице `m`** одна строка соответствует одному календарному месяцу и содержит показатели по всей подписной базе: количество подписок на начало и конец месяца, новые оформления и завершения.

**Почему выбрали именно такой уровень?** Мы исследуем, как меняется подписная база во времени и почему замедляется ее прирост. Поэтому нам важно сравнивать общие показатели по месяцам, а не анализировать каждую подписку или клиента отдельно. При этом одна и та же подписка может учитываться в базе нескольких месяцев, если продолжает действовать, мы считаем количество действующих подписок на определенную дату, а не уникальные оформления за весь период.

**3. Период исследования**

Анализируем 18 полных месяцев - с февраля 2025 по июль 2026 года. Январь 2025 года исключаем, поскольку данные начинаются только с 6 января. Август 2026 года тоже не используем, так как он неполный и содержит массовые завершения, отдельно рассмотренные в курсе. Используем все полные месяцы между этими границами, чтобы увидеть долгосрочную динамику, а не делать выводы по нескольким отдельным месяцам. Для сравнения оформлений, завершений и прироста рассчитываем средние значения в день. Это позволяет корректно сравнивать месяцы разной длины, например февраль и июль.

**4. Группы подписок (сегменты)**

Для расчетов разделяем подписки на две группы:

- **Стартовая база** - подписки, которые уже действовали на начало месяца
- **Новые оформления** - подписки, которые появились в течение месяца

Такое разделение необходимо для правильного расчета доли завершений. Например, если подписка оформилась и завершилась в одном месяце, ее учитываем в общих оформлениях и завершениях, но не включаем в расчет доли завершений стартовой базы.

**5. Правила расчета**

Для определения начала и завершения подписки используем даты `started_at` и `ended_at`. Все расчеты выполняем в UTC. Базу на начало месяца определяем как количество подписок, которые были оформлены раньше начала месяца и еще действовали непосредственно перед этой датой. Аналогично рассчитываем базу на конец месяца. Если дата завершения отсутствует, считаем, что подписка продолжает действовать. Оформления и завершения, произошедшие точно на границе месяца, относим к наступившему месяцу.

База на конец = База на начало + Новые оформления − Завершения

## Подключение

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sqlalchemy import text

from prime.config import get_engine
from prime.factors import contrib, waterfall

#настройки отображения таблиц
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

#единый стиль графиков
sns.set_theme(
    style="whitegrid",
    context="notebook",
    font="DejaVu Sans")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F8FAFC",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": "#E2E8F0",
    "grid.linewidth": 0.7,
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlepad": 18})


#подключение к базе (по образцу семинаров)
engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить SQL-запрос и вернуть DataFrame."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("Подключились как:", q("select current_user as login").iloc[0, 0])


## Расчет месячных показателей подписной базы

Для анализа изменения подписной базы нужно определить, сколько подписок действовало на начало и конец каждого месяца, сколько новых подписок было оформлено и сколько завершилось. Так мы сможем рассчитать чистый прирост подписок и понять, за счет чего меняется скорость роста базы

Сначала проверяем исходные данные: отдельным SQL-запросом определяем общее количество подписок, наличие некорректных дат, а также количество оформлений и завершений за весь исследуемый период, потом эти значения используются для проверки месячных расчетов

Потои рассчитываем показатели по месяцам: для каждого из 18 месяцев выполняем один SQL-запрос к `prime.subscriptions`, который определяет:
- Базу на начало и конец месяца — количество действующих подписок на соответствующие даты
- Новые оформления — подписки, начавшиеся в течение месяца
- Завершения — подписки, закончившиеся в течение месяца. Отдельно выделяем завершения подписок, существовавших на начало месяца, и подписок, оформленных в этом же месяце
- Причины завершений — распределение завершившихся подписок по значениям `cancel_reason` 

Результат: объединяем месячные расчеты в таблицу `m`, где одна строка соответствует одному месяцу, а столбцы содержат рассчитанные показатели. Эта таблица используем далее для анализа динамики, расчета метрик и факторной декомпозиции

In [ ]:

#задаем период исследования - 18 полных месяцев
START = '2025-02-01'
END = '2026-08-01'


#первый запрос - проверяем исходные данные за весь период
AUDIT_SQL = """
select
    count(*) as subscriptions, --всего подписок в таблице
    count(*) filter(
        where started_at is null or ended_at <= started_at
    ) as invalid_dates, --подписки с некорректными датами
    count(*) filter(
        where started_at >= cast(:a as timestamptz)
          and started_at < cast(:b as timestamptz)
    ) as starts_in_period, --оформления за весь период
    count(*) filter(
        where ended_at >= cast(:a as timestamptz)
          and ended_at < cast(:b as timestamptz)
    ) as ends_in_period --завершения за весь период
from prime.subscriptions
"""

#выполняем запрос и показываем результаты проверки
quality = q(AUDIT_SQL, a=START, b=END)
display(quality)

#проверяем, что в таблице нет некорректных дат
assert quality.loc[0, 'invalid_dates'] == 0, 'Обнаружены некорректные даты'


#второй запрос - считаем показатели за один месяц
#:a - начало месяца, :b - начало следующего месяца
MONTH_SQL = """
select
    count(*) filter(
        where started_at < cast(:a as timestamptz)
    ) as base_start, --действующие подписки на начало месяца

    count(*) filter(
        where started_at >= cast(:a as timestamptz)
    ) as new_starts, --новые оформления за месяц

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
    ) as cancellations, --все завершения за месяц

    count(*) filter(
        where started_at < cast(:a as timestamptz)
          and ended_at < cast(:b as timestamptz)
    ) as cancellations_base, --завершения из стартовой базы

    count(*) filter(
        where started_at >= cast(:a as timestamptz)
          and ended_at < cast(:b as timestamptz)
    ) as cancellations_new, --завершения новых подписок

    count(*) filter(
        where ended_at is null
           or ended_at >= cast(:b as timestamptz)
    ) as base_end, --действующие подписки на конец месяца

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and cancel_reason = 'not_using'
    ) as not_using, --причина: не используют

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and cancel_reason = 'too_expensive'
    ) as too_expensive, --причина: дорого

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and cancel_reason = 'payment_failed'
    ) as payment_failed, --причина: проблема оплаты

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and cancel_reason = 'switched'
    ) as switched, --причина: перешли к другому

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and cancel_reason = 'other'
    ) as other, --другая причина

    count(*) filter(
        where ended_at < cast(:b as timestamptz)
          and coalesce(cancel_reason, 'unknown') not in
          ('not_using', 'too_expensive', 'payment_failed', 'switched', 'other')
    ) as unknown_reason --неизвестная или отсутствующая причина

from prime.subscriptions
where started_at < cast(:b as timestamptz)
  and (ended_at is null or ended_at >= cast(:a as timestamptz))
"""


#создаем список границ месяцев от февраля 2025 до августа 2026
months = pd.date_range(START, END, freq='MS')

#создаем пустой список для результатов
parts = []

#перебираем месяцы по очереди
for a, b in zip(months[:-1], months[1:]):

    #выполняем запрос за очередной месяц
    part = q(MONTH_SQL, a=str(a.date()), b=str(b.date()))

    #добавляем название месяца
    part['month'] = a.strftime('%Y-%m')

    #добавляем количество дней в месяце
    part['days'] = (b - a).days

    #сохраняем результат месяца
    parts.append(part)

#объединяем результаты всех месяцев в одну таблицу
m = pd.concat(parts, ignore_index=True).set_index('month')

#приводим показатели к целым числам
m = m.astype('int64')

#проверяем количество месяцев
print('Полных месяцев:', len(m))

## Расчет метрик и проверка баланса подписок

В предыдущем разделе мы получили таблицу `m` с месячными показателями подписной базы. Теперь проверим корректность этих данных и рассчитаем метрики для анализа динамики роста

1. Проверка корректности данных

Проверяем, что для каждого месяца выполняется баланс:

База на конец = База на начало + Новые оформления − Завершения

Также проверяем, что:
- Все завершения разделены на завершения подписок из стартовой базы и новых оформлений
- Сумма завершений по причинам совпадает с общим количеством завершений
- База на конец одного месяца равна базе на начало следующего
- Суммы месячных оформлений и завершений совпадают с результатами контрольного SQL-запроса за весь период

Проверки выполняются с помощью `assert`: если какое-либо равенство нарушено, выполнение кода останавливается с ошибкой.

2. Расчет основных метрик

- **Чистый прирост** = оформления − завершения. Показывает, на сколько подписок увеличилась или уменьшилась база за месяц
- **Прирост в день** = чистый прирост / число дней в месяце. Это основная метрика проекта, позволяющая сравнивать скорость роста в месяцах разной длины
- **Прирост, %** = чистый прирост / база на начало × 100%. Показывает относительный рост базы
- **Оформления и завершения в день** = соответствующее количество событий / число дней. Позволяют отдельно анализировать приток и завершения
- **Доля завершений стартовой базы** = завершения подписок, существовавших на начало месяца / стартовая база × 100%. Новые оформления в этот показатель не входят
- **Доля завершений на 30 суток** = месячная доля завершений × 30 / число дней в месяце. Это нормировка для сопоставления месяцев
- **Завершения / оформления, %** = все завершения / новые оформления × 100%. Показывает, какая часть притока количественно компенсируется завершениями

Дополнительно рассчитываем завершения новых оформлений в день и частоту завершений стартовой базы на один день. Эти показатели понадобятся для факторной декомпозиции

Результат: таблица `m` дополняется рассчитанными метриками, которые далее используются для построения графиков и объяснения изменения прироста подписной базы

In [ ]:

#задаем понятные названия причин завершения подписок
REASONS = {
    'not_using': 'Не используют',
    'too_expensive': 'Дорого',
    'payment_failed': 'Проблема оплаты',
    'switched': 'Перешли к другому',
    'other': 'Другая',
    'unknown_reason': 'Не указана/неизвестна'}


#1. проверяем корректность исходных показателей
#проверяем, что получили 18 месяцев и все значения неотрицательные
assert len(m) == 18
assert m.ge(0).all().all()
#проверяем, что знаменатели для расчета долей не равны нулю
assert m.base_start.gt(0).all()
assert m.new_starts.gt(0).all()
#проверяем баланс подписок за каждый месяц
assert (m.base_start + m.new_starts - m.cancellations == m.base_end).all()
#проверяем, что все завершения разделены на старые и новые подписки
assert (m.cancellations_base + m.cancellations_new == m.cancellations).all()
#проверяем, что завершений не больше, чем подписок в соответствующих группах
assert m.cancellations_base.le(m.base_start).all()
assert m.cancellations_new.le(m.new_starts).all()
#проверяем, что сумма завершений по причинам совпадает с общим числом
assert (m[list(REASONS)].sum(axis=1) == m.cancellations).all()
#проверяем, что конец одного месяца совпадает с началом следующего
assert np.array_equal(
    m.base_end.iloc[:-1].to_numpy(),
    m.base_start.iloc[1:].to_numpy())
#сверяем месячные суммы с независимым контрольным запросом
assert m.new_starts.sum() == quality.loc[0, 'starts_in_period']
assert m.cancellations.sum() == quality.loc[0, 'ends_in_period']
print('Все проверки данных пройдены')


#2. рассчитываем метрики для анализа динамики
#считаем чистый прирост подписок за месяц
m['net_growth'] = m.new_starts - m.cancellations
#считаем процент прироста относительно базы на начало месяца
m['growth_pct'] = 100 * m.net_growth / m.base_start
#считаем среднее количество новых оформлений в день
m['new_per_day'] = m.new_starts / m.days
#считаем среднее количество завершений в день
m['cancel_per_day'] = m.cancellations / m.days
#считаем чистый прирост в день - основная метрика проекта
m['net_per_day'] = m.net_growth / m.days
#считаем месячную долю завершений стартовой базы в процентах
m['base_cancel_pct'] = 100 * m.cancellations_base / m.base_start
#приводим долю завершений к условному месяцу из 30 дней
m['base_cancel_30_pct'] = m.base_cancel_pct * 30 / m.days
#считаем отношение всех завершений к новым оформлениям в процентах
m['cancellations_to_new_pct'] = 100 * m.cancellations / m.new_starts
#считаем завершения новых оформлений в среднем за день
m['new_cancel_per_day'] = m.cancellations_new / m.days
#считаем частоту завершений стартовой базы на один календарный день
m['base_cancel_per_sub_day'] = m.cancellations_base / (m.base_start * m.days)
#проверяем, что общий прирост равен изменению базы за весь период
assert m.net_growth.sum() == m.base_end.iloc[-1] - m.base_start.iloc[0]


#3. выводим итоговую таблицу с основными метриками

#задаем русские названия столбцов для вывода
LABELS = {
    'base_start': 'База на начало',
    'new_starts': 'Оформления',
    'cancellations': 'Завершения',
    'base_end': 'База на конец',
    'net_growth': 'Прирост',
    'new_per_day': 'Оформления в день',
    'cancel_per_day': 'Завершения в день',
    'net_per_day': 'Прирост в день',
    'base_cancel_30_pct': 'Завершения стартовой базы на 30 суток, %',
    'cancellations_to_new_pct': 'Завершения / оформления, %'}

display(m[list(LABELS)].rename(columns=LABELS))


## Динамика показателей подписной базы

Анализируем изменение основных показателей за 18 полных месяцев - с февраля 2025 по июль 2026.

Для количества действующих подписок используем значения на конец каждого месяца. Количество оформлений, завершений и чистый прирост приводим к среднему значению за день, чтобы сравнение не зависело от разной длины месяцев. Долю завершений стартовой базы дополнительно нормируем к 30 суткам.

Графики построены с помощью Seaborn и Matplotlib 

In [ ]:

#создаем позиции для всех 18 месяцев
x = np.arange(len(m))

#задаем общую палитру графиков
BLUE = '#164B72'
GRAY = '#64748B'
LIGHT = '#AAB8C5'
TEAL = '#2A9D8F'

#создаем общую функцию оформления оси месяцев
def month_axis(ax):
    #размещаем отметки каждого месяца
    ax.set_xticks(x)
    #подписываем месяцы в формате месяц.год
    ax.set_xticklabels(
        [pd.Period(v).strftime('%m.%y') for v in m.index],
        rotation=45,
        ha='right',
        fontsize=8.5,
        color=GRAY)
    #подписываем горизонтальную ось
    ax.set_xlabel('Месяц, UTC', labelpad=12, color=GRAY)
    #оставляем только горизонтальные линии сетки
    ax.grid(axis='y', color='#E6ECF0', linewidth=0.8)
    ax.grid(axis='x', visible=False)
    #убираем лишние черточки на осях
    ax.tick_params(axis='both', length=0, pad=7)
    #оставляем небольшие отступы от краев
    ax.set_xlim(-0.6, len(m) - 0.4)

#создаем функцию вывода графиков
def show_chart(fig):
    #настраиваем отступы, если способ размещения еще не задан
    if fig.get_layout_engine() is None:
        fig.tight_layout()
    #показываем график
    plt.show()

#1. количество действующих подписок на конец месяца
#переводим количество подписок в тысячи
values = m['base_end'] / 1000
#создаем область графика
fig, ax = plt.subplots(figsize=(13, 5.2))
#задаем белый фон
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
#строим линию динамики с помощью seaborn
sns.lineplot(
    x=x,
    y=values,
    ax=ax,
    color=BLUE,
    linewidth=2.6,
    marker='o',
    markersize=5)
#выделяем последнюю точку другим цветом
ax.scatter(
    x[-1],
    values.iloc[-1],
    s=75,
    color=TEAL,
    edgecolor='white',
    linewidth=1.5,
    zorder=5)

#подписываем значение за каждый месяц
for i, value in enumerate(values):
    #последнее значение выделяем зеленым
    color = TEAL if i == len(values) - 1 else BLUE
    #размещаем небольшую подпись над точкой
    ax.annotate(
        f'{value:.1f}'.replace('.', ','),
        (x[i], value),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=8,
        color=color,
        fontweight='bold' if i in (0, len(values)-1) else 'normal')

#задаем название графика
ax.set_title(
    'Действующие подписки на конец месяца',
    loc='left',
    fontsize=15,
    fontweight='bold',
    color='#1F2937',
    pad=26)

#подписываем вертикальную ось
ax.set_ylabel('Тыс. подписок', labelpad=13, color=GRAY)
#задаем границы вертикальной оси от нуля
ax.set_ylim(0, 230)
ax.set_yticks(np.arange(0, 201, 50))
#настраиваем подписи вертикальной оси
ax.tick_params(axis='y', labelsize=9, colors=GRAY)
#применяем общее оформление месяцев
month_axis(ax)
#убираем рамку вокруг графика
sns.despine(ax=ax, left=True, bottom=True)
#выводим готовый график
show_chart(fig)


**Вывод по графику 1.** Количество действующих подписок на конец месяца увеличилось со 107,7 тыс. в феврале 2025 года до 192,3 тыс. в июле 2026 года (+78,5%). База росла на протяжении всего исследуемого периода, но это не означает, что скорость ее прироста оставалась неизменной

In [ ]:
#2. оформления, завершения и чистый прирост в день

#создаем короткие ссылки на ряды
starts = m['new_per_day']
ends = m['cancel_per_day']
net = m['net_per_day']
#создаем область графика
fig, ax = plt.subplots(figsize=(13, 5.4))
#задаем белый фон
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
#рисуем три линии
sns.lineplot(
    x=x,
    y=starts,
    ax=ax,
    color=GRAY,
    linewidth=2.0,
    marker='o',
    markersize=4.5,
    label='Оформления')

sns.lineplot(
    x=x,
    y=ends,
    ax=ax,
    color=TEAL,
    linewidth=2.0,
    marker='o',
    markersize=4.5,
    label='Завершения')

sns.lineplot(
    x=x,
    y=net,
    ax=ax,
    color=BLUE,
    linewidth=2.8,
    marker='o',
    markersize=5,
    label='Чистый прирост')

#слегка выделяем линию чистого прироста заливкои
ax.fill_between(x, net, 0, color='#DDEBF3', alpha=0.22)

#выделяем последние точки
ax.scatter(x[-1], starts.iloc[-1], s=60, color=GRAY, edgecolor='white', linewidth=1.2, zorder=5)
ax.scatter(x[-1], ends.iloc[-1], s=60, color=TEAL, edgecolor='white', linewidth=1.2, zorder=5)
ax.scatter(x[-1], net.iloc[-1], s=70, color=BLUE, edgecolor='white', linewidth=1.2, zorder=5)

#подписываем значения по всем месяцам для оформлений
for i, value in enumerate(starts):
    ax.annotate(
        f'{value:.0f}',
        (x[i], value),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=7.5,
        color=GRAY)

#подписываем значения по всем месяцам для завершений
for i, value in enumerate(ends):
    ax.annotate(
        f'{value:.0f}',
        (x[i], value),
        xytext=(0, -13),
        textcoords='offset points',
        ha='center',
        va='top',
        fontsize=7.5,
        color=TEAL)

#подписываем значения по всем месяцам для чистого прироста
for i, value in enumerate(net):
    ax.annotate(
        f'{value:.0f}',
        (x[i], value),
        xytext=(0, 11),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=7.6,
        color=BLUE,
        fontweight='bold' if i in (0, len(net) - 1) else 'normal')

#задаем название графика
ax.set_title(
    'Оформления, завершения и чистый прирост в день',
    loc='left',
    fontsize=15,
    fontweight='bold',
    color='#1F2937',
    pad=26)

#подписываем вертикальную ось
ax.set_ylabel('Подписок в среднем за день', labelpad=13, color=GRAY)

#задаем границы вертикальнои оси
ax.set_ylim(0, 420)
ax.set_yticks(np.arange(0, 421, 50))
ax.tick_params(axis='y', labelsize=9, colors=GRAY)

#оформляем легенду
ax.legend(
    ncol=3,
    loc='upper left',
    bbox_to_anchor=(0, 1.02),
    frameon=False,
    fontsize=9)

#применяем общее оформление оси месяцев
month_axis(ax)
#убираем лишние рамки
sns.despine(ax=ax, left=True, bottom=True)
#показываем график
show_chart(fig)

**Вывод по графику 2.** Среднее число новых оформлений в день на протяжении периода меняется слабо и остается примерно в диапазоне 354-370. При этом среднее число завершений в день заметно растет - со 137 в феврале 2025 года до 239 в июле 2026 года, поэтому чистый прирост снижается с 233 до 127 подписок в день

In [ ]:
#3. доля завершений стартовои базы на 30 суток

#берем ряд для графика
values = m['base_cancel_30_pct']
#создаем область графика
fig, ax = plt.subplots(figsize=(13, 5.2))
#задаем белыи фон
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
#рисуем горизонтальныи ориентир на уровне 4 процентов
ax.axhline(
    4,
    color=LIGHT,
    linewidth=1.1,
    linestyle='--',
    zorder=0)
#строим линию динамики
sns.lineplot(
    x=x,
    y=values,
    ax=ax,
    color=BLUE,
    linewidth=2.6,
    marker='o',
    markersize=5)
#выделяем последнюю точку
ax.scatter(
    x[-1],
    values.iloc[-1],
    s=75,
    color=TEAL,
    edgecolor='white',
    linewidth=1.5,
    zorder=5)
#подписываем значение за каждыи месяц
for i, value in enumerate(values):
    #последнее значение выделяем зеленым
    color = TEAL if i == len(values) - 1 else BLUE
    #размещаем подпись над точкои
    ax.annotate(
        f'{value:.2f}'.replace('.', ',') + '%',
        (x[i], value),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=7.8,
        color=color,
        fontweight='bold' if i in (0, len(values) - 1) else 'normal')

#задаем название графика
ax.set_title(
    'Доля завершений стартовои базы на 30 суток',
    loc='left',
    fontsize=15,
    fontweight='bold',
    color='#1F2937',
    pad=26)

#подписываем вертикальную ось
ax.set_ylabel('Завершения стартовои базы, %', labelpad=13, color=GRAY)
#задаем границы и деления вертикальнои оси
ax.set_ylim(0, 5.1)
ax.set_yticks(np.arange(0, 5.1, 0.5))
ax.tick_params(axis='y', labelsize=9, colors=GRAY)
#применяем общее оформление оси месяцев
month_axis(ax)
#убираем лишние рамки
sns.despine(ax=ax, left=True, bottom=True)
#показываем график
show_chart(fig)

**Вывод по графику 3.** Нормированная доля завершений стартовой базы в течение периода не показывает общего роста и колеблется вблизи 4%. В феврале 2025 года показатель составлял 4,06%, а в июле 2026 года - 3,80%, то есть увеличение абсолютного числа завершений связано скорее с ростом самой базы, чем с общим ухудшением удержания

Итого: Рост числа завершений объясняется не только изменением их доли: важен размер базы, к которой она применяется. Это проверяет следующая декомпозиция.


## Факторная декомпозиция изменения прироста

В предыдущем разделе мы установили, что чистый прирост подписной базы снизился с 233 до 127 подписок в день. Теперь определим, за счет каких факторов произошло это изменение.

Используем метод цепных подстановок. Он позволяет разложить изменение итогового показателя на вклады отдельных факторов, измеренные в тех же единицах.

Чистый прирост за месяц равен разнице между оформлениями и завершениями. Для сравнения месяцев разной длины используем показатель прироста в день:

**`g = p − B × r − u`**, где:

- `g` - чистый прирост подписок в день
- `p` - оформления в день
- `B` - действующие подписки на начало месяца
- `r` - завершения стартовой базы / (стартовая база × число дней)
- `u` - завершения новых оформлений в день

Таким образом, изменение прироста зависит от 4 факторов: притока новых оформлений, размера стартовой базы, частоты завершений стартовой базы и завершений новых оформлений.

Для произведения `B × r` применяем метод цепных подстановок: сначала изменяем размер базы, затем частоту завершений. Используем функцию `contrib` из `prime.factors`, которая рассчитывает вклад каждого фактора без округления.

Завершения учитываются с отрицательным знаком, поскольку уменьшают чистый прирост. Сумма всех факторных вкладов должна совпадать с фактическим изменением прироста.

Порядок расчета

Сначала рассчитываем факторные вклады для каждой пары соседних месяцев, чтобы проследить динамику изменений. Затем отдельно сравниваем февраль 2025 и июль 2026 года, чтобы определить факторы общего замедления прироста за исследуемый период.

Результаты представляем в подписках в день. Декомпозиция показывает арифметические вклады факторов, но не доказывает причинное влияние конкретных управленческих решений

In [ ]:
#создаем функцию для разложения изменения прироста между двумя месяцами
def decompose(a, b):
    #задаем размер базы и частоту завершений в первом месяце
    before = {
        'base': a.base_start,
        'rate': a.base_cancel_per_sub_day}
    #задаем те же показатели во втором месяце
    after = {
        'base': b.base_start,
        'rate': b.base_cancel_per_sub_day}
    
    #считаем влияние базы и частоты завершений методом цепных подстановок
    #сначала изменяем размер базы, затем частоту завершений
    factors = contrib(before, after, order=['base', 'rate'])

    #собираем вклады всех четырех факторов в изменение прироста
    result = pd.Series({
        'Оформления': b.new_per_day - a.new_per_day,
        'Размер стартовой базы': -factors['base'],
        'Частота завершений стартовой базы': -factors['rate'],
        'Завершения новых оформлений': -(b.new_cancel_per_day - a.new_cancel_per_day)})

    #проверяем, что сумма вкладов равна фактическому изменению прироста
    assert np.isclose(
        result.sum(),
        b.net_per_day - a.net_per_day,
        atol=1e-8)

    #возвращаем рассчитанные вклады
    return result


#1. рассчитываем изменения между соседними месяцами

#создаем список для результатов
monthly_effects = []
#сравниваем каждый месяц с предыдущим
for i in range(1, len(m)):
    #получаем показатели предыдущего и текущего месяца
    previous = m.iloc[i - 1]
    current = m.iloc[i]
    #рассчитываем факторные вклады и сохраняем результат
    monthly_effects.append(decompose(previous, current))

#собираем результаты в таблицу по месяцам
adjacent = pd.DataFrame(monthly_effects, index=m.index[1:])
#добавляем фактическое изменение прироста для сравнения
adjacent['Изменение прироста в день'] = m.net_per_day.diff().iloc[1:]
#показываем результаты с округлением до двух знаков
display(adjacent.round(2))


#2. рассчитываем изменение между первым и последним месяцами
#сравниваем февраль 2025 и июль 2026
effects = decompose(m.iloc[0], m.iloc[-1])
#выводим факторные вклады за исследуемый период
print(f'Сравнение: {m.index[0]} - {m.index[-1]}')
display(effects.to_frame('Вклад, подписок в день').round(2))
#проверяем и показываем итоговое изменение прироста
print(f'Сумма вкладов: {effects.sum():.2f}')
print(f'Фактическое изменение: {m.net_per_day.iloc[-1] - m.net_per_day.iloc[0]:.2f}')


**Вывод по факторной декомпозиции.** При сравнении соседних месяцев увеличение стартовой базы каждый раз вносило отрицательный вклад в изменение прироста, а влияние остальных факторов менялось. За весь исследуемый период чистый прирост снизился на 106,05 подписки в день. Основной отрицательный вклад внесло увеличение стартовой базы (−117,87), а снижение частоты завершений частично компенсировало его (+16,24). Изменение притока оформлений (−3,94) и завершений новых подписок (−0,48) повлияло намного меньше

In [ ]:
#4. факторная декомпозиция изменения чистого прироста

#копируем рассчитанные вклады для графика
short_effects = effects.copy()
#сокращаем названия факторов для подписей
short_effects.index = [
    'Приток\nоформлений',
    'Размер\nстартовой базы',
    'Частота завершений\nстартовой базы',
    'Завершения\nновых оформлений']
#строим каскадную диаграмму функцией из семинара
ax = waterfall(
    short_effects,
    title='Факторная декомпозиция изменения чистого прироста')
#получаем фигуру и увеличиваем ее размер
fig = ax.figure
fig.set_size_inches(13, 5.5)
#задаем белый фон
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
#задаем цвета факторов - отрицательные синие, положительные зеленые
for patch, value in zip(ax.patches[:-1], effects):
    patch.set_facecolor(TEAL if value > 0 else BLUE)
#выделяем итоговый столбик серым цветом
ax.patches[-1].set_facecolor(GRAY)
#задаем прозрачность и границы столбиков
for patch in ax.patches:
    patch.set_alpha(0.9)
    patch.set_linewidth(0)
#подписываем каждый фактор и общий итог с одним знаком после запятой
values = [*effects.to_list(), effects.sum()]

for label, value in zip(ax.texts, values):
    label.set_text(f'{value:+.1f}'.replace('.', ','))
    label.set_fontsize(10)
    label.set_fontweight('bold')
    label.set_color('#1F2937')

#задаем название графика
ax.set_title(
    'Факторная декомпозиция изменения чистого прироста',
    loc='left',
    fontsize=15,
    fontweight='bold',
    color='#1F2937',
    pad=26)

#подписываем вертикальную ось
ax.set_ylabel('Вклад, подписок в день', labelpad=13, color=GRAY)
#указываем сравниваемые месяцы
ax.set_xlabel(
    'Сравнение: февраль 2025 - июль 2026',
    labelpad=18,
    color=GRAY)

#настраиваем подписи факторов
ax.tick_params(axis='x', labelsize=9, length=0, pad=10)
ax.tick_params(axis='y', labelsize=9, colors=GRAY, length=0)
#определяем нижнюю границу с учетом всех накопленных вкладов
lowest = min(0, effects.cumsum().min(), effects.sum())
#оставляем достаточно места для подписей
ax.set_ylim(lowest - 25, 18)
#оставляем только горизонтальную сетку
ax.grid(axis='y', color='#E6ECF0', linewidth=0.8)
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
#выделяем нулевую линию
ax.axhline(0, color=GRAY, linewidth=1)
#убираем лишние рамки графика
sns.despine(ax=ax, left=True, bottom=True)
#показываем график
show_chart(fig)


**Вывод по графику 4.** Основной отрицательный вклад в изменение чистого прироста внесло увеличение стартовой базы (−117,9 подписки в день), а снижение частоты завершений частично компенсировало этот эффект (+16,2). Изменения притока оформлений и завершений новых подписок оказали значительно меньшее влияние. 

Итого, при почти постоянном притоке рост базы сопровождается увеличением абсолютного числа завершений, даже без ухудшения наблюдаемой частоты ухода

## Дополнительная проверка: сравнение трехмесячных периодов

В факторной декомпозиции мы сравнили первый и последний месяцы исследования - февраль 2025 и июль 2026 года. Но отдельные месяцы могут отличаться из-за временных колебаний показателей. Поэтому дополнительно проверим, сохраняется ли вывод о замедлении прироста при сравнении более длительных периодов.

Для этого сравним первые три месяца (февраль - апрель 2025 года) и последние три месяца (май - июль 2026 года)

Для каждого периода рассчитаем среднее число оформлений, завершений и чистый прирост в день. Для этого суммируем соответствующие показатели за три месяца и разделим на общее количество календарных дней. Такой подход учитывает разную длину месяцев.

Если средний чистый прирост за последние три месяца также окажется ниже, чем за первые три, это подтвердит, что замедление наблюдается не только при сравнении двух отдельных месяцев

In [ ]:
#сравнение первых и последних трех месяцев

#задаем два периода для сравнения
periods = {
    'Первые 3 месяца': m.iloc[:3],
    'Последние 3 месяца': m.iloc[-3:]}
#создаем пустой список для результатов
window_rows = []
#перебираем два трехмесячных периода
for name, period in periods.items():
    #считаем общее количество дней за три месяца
    days = period['days'].sum()
    #рассчитываем средние показатели в день за весь период
    window_rows.append({
        'Окно': name,
        'Период': f'{period.index[0]} - {period.index[-1]}',
        'Оформления в день': period['new_starts'].sum() / days,
        'Завершения в день': period['cancellations'].sum() / days,
        'Прирост в день': period['net_growth'].sum() / days})
    
#объединяем результаты в таблицу
windows = pd.DataFrame(window_rows).set_index('Окно')
#показываем таблицу с округлением до двух знаков
display(windows.round(2))
#получаем средний прирост в день за первый и последний периоды
first = windows.loc['Первые 3 месяца', 'Прирост в день']
last = windows.loc['Последние 3 месяца', 'Прирост в день']
#считаем изменение прироста между периодами в процентах
window_change = 100 * (last / first - 1)
#выводим результат сравнения
print(f'Изменение прироста между периодами: {window_change:.1f}%')

**Вывод** Средний чистый прирост снизился с 219,39 подписки в день в начале исследования до 126,84 в конце (-42,2%). При этом приток новых оформлений практически не изменился (364,85 против 363,01 в день), а количество завершений выросло со 145,46 до 236,17 в день. Таким образом, сравнение трехмесячных периодов подтверждает, что замедление прироста связано преимущественно с увеличением числа завершений, а не с сокращением притока новых подписок

## Анализ причин завершения подписок

В предыдущих разделах мы установили, что абсолютное количество завершений увеличивается по мере роста подписной базы. Теперь рассмотрим причины завершений, зафиксированные в исходных данных. Для этого сгруппируем все завершения за исследуемые 18 месяцев по значениям `cancel_reason` и рассчитаем долю каждой причины в общем количестве завершений.

**Доля причины = Количество завершений по данной причине / Общее количество завершений × 100%**

Доли рассчитываем по общим количествам за весь период, а не как среднее месячных процентов. Это позволяет учитывать все завершения независимо от того, в каком месяце они произошли. Результаты представим в таблице и на горизонтальной столбчатой диаграмме. Категорию с неизвестной или отсутствующей причиной сохраним в таблице, чтобы проверить полноту данных.

Анализ записанных причин не позволяет установить причинно-следственные связи, но помогает определить направления для дальнейшего изучения удержания подписчиков.

In [ ]:
#5.анализ причин завершения подписок

#суммируем количество завершений по каждой причине за весь период
reason_totals = m[list(REASONS)].sum()
#заменяем технические названия причин на русские
reason_totals = reason_totals.rename(index=REASONS)
#сортируем причины по убыванию количества завершений
reason_totals = reason_totals.sort_values(ascending=False)
#создаем таблицу с количеством завершений и долей каждой причины
reason_table = pd.DataFrame({
    'Завершений': reason_totals,
    'Доля, %': 100 * reason_totals / m.cancellations.sum()})
#показываем таблицу, округляя доли до двух знаков
display(reason_table.round(2))


#строим график распределения причин завершения
#оставляем для графика только причины с ненулевым количеством завершений
shown = reason_table[reason_table['Завершений'] > 0]
#создаем область графика
fig, ax = plt.subplots(figsize=(13, 5.2))
#задаем белый фон
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
#строим горизонтальные столбики с помощью seaborn
sns.barplot(
    x=shown['Доля, %'],
    y=shown.index,
    color=LIGHT,
    ax=ax)
#выделяем наиболее частую причину синим цветом
for patch, name in zip(ax.patches, shown.index):
    patch.set_facecolor(BLUE if name == shown.index[0] else LIGHT)
#подписываем долю каждой причины рядом со столбиком
ax.bar_label(
    ax.containers[0],
    labels=[f'{value:.1f}%'.replace('.', ',') for value in shown['Доля, %']],
    padding=8,
    fontsize=9,
    color='#1F2937',
    fontweight='bold')
#задаем нейтральное название графика
ax.set_title(
    'Распределение завершений подписок по причинам',
    loc='left',
    fontsize=15,
    fontweight='bold',
    color='#1F2937',
    pad=26)
#подписываем горизонтальную ось
ax.set_xlabel('Доля всех завершений, %', labelpad=13, color=GRAY)
#убираем название вертикальной оси
ax.set_ylabel('')
#задаем границы горизонтальной оси с местом для подписей
ax.set_xlim(0, 42)
ax.set_xticks(np.arange(0, 41, 10))
#настраиваем подписи осей
ax.tick_params(axis='x', labelsize=9, colors=GRAY, length=0)
ax.tick_params(axis='y', labelsize=10, colors='#1F2937', length=0, pad=12)
#оставляем только вертикальные линии сетки
ax.grid(axis='x', color='#E6ECF0', linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
#убираем лишние рамки
sns.despine(ax=ax, left=True, bottom=True)
#показываем готовый график
show_chart(fig)

**Вывод по графику 5.** Наиболее распространенные причины завершения подписок - «Не используют» (34,0%) и «Дорого» (24,2%). Совокупно на них приходится 58,2% всех завершений за исследуемый период. Поэтому для улучшения удержания в первую очередь стоит изучить причины низкой активности подписчиков и оценить, насколько стоимость подписки соответствует воспринимаемой клиентами ценности. Но записанные причины не позволяют напрямую объяснить замедление прироста подписной базы

## Рекомендации и сценарий планирования роста

**Основной вывод** За исследуемый период чистый прирост подписной базы заметно замедлился, несмотря на практически неизменный приток новых оформлений. Факторная декомпозиция показала, что основной отрицательный вклад связан с увеличением размера стартовой базы: при сопоставимой частоте завершений большее количество действующих подписок приводит к большему абсолютному числу завершений. При этом нормированная доля завершений стартовой базы не показывает общего роста. Следовательно, замедление чистого прироста не следует автоматически интерпретировать как ухудшение удержания подписчиков.

**Рекомендации**

1. Учитывать завершения при планировании роста: устанавливать цели не только по количеству новых оформлений, но и по чистому приросту подписной базы с учетом ее текущего размера и ожидаемых завершений

2. Контролировать показатели в комплексе: ежемесячно отслеживать оформления в день, завершения в день, чистый прирост и долю завершений стартовой базы. Это позволит различать увеличение абсолютного числа завершений из-за роста базы и изменение относительной частоты завершений.

3. Изучить основные причины завершений: проверить, почему подписчики перестают пользоваться сервисом (34,0% завершений) или считают его дорогим (24,2%). На основе дополнительной диагностики можно разработать меры по удержанию и проверить их эффективность в эксперименте. 

**Простой сценарий планирования** Дополнительно рассчитаем, сколько новых оформлений в день потребовалось бы для восстановления скорости чистого прироста февраля 2025 года при сохранении количества завершений на уровне июля 2026 года. Для этого сложим целевой чистый прирост и среднее число завершений в день за июль. Полученное значение сравним с фактическим количеством новых оформлений в июле. Такой расчет позволяет оценить необходимый масштаб притока при заданном уровне завершений. Это условный сценарий: в будущем количество завершений может изменяться. Без данных о стоимости привлечения и удержания нельзя определить, какой способ достижения целевого прироста экономически выгоднее

In [ ]:
#сценарий восстановления прежней скорости прироста

#берем прирост в день за февраль 2025 как целевой показатель
target_growth = m.net_per_day.iloc[0]
#берем среднее число завершений в день за июль 2026
current_cancellations = m.cancel_per_day.iloc[-1]
#берем фактическое число оформлений в день за июль 2026
current_inflow = m.new_per_day.iloc[-1]
#считаем, сколько оформлений нужно для достижения цели
#предполагаем, что количество завершений не меняется
required_inflow = target_growth + current_cancellations
#считаем, насколько нужно увеличить текущий приток в процентах
inflow_change = 100 * (required_inflow / current_inflow - 1)
#показываем исходные показатели и результат сценария
print(f'Целевой прирост: {target_growth:.1f} подписки в день')
print(f'Текущий приток: {current_inflow:.1f} оформлений в день')
print(f'Текущие завершения: {current_cancellations:.1f} в день')
print(f'Необходимый приток: {required_inflow:.1f} оформлений в день')
print(f'Увеличение притока: {inflow_change:.1f}%')


Для восстановления скорости чистого прироста февраля 2025 года при сохранении июльского уровня завершений потребовалось бы около 472 новых оформлений в день вместо текущих 366 (+29,0%). Это показывает, что по мере увеличения подписной базы для поддержания прежнего темпа роста требуется либо больший приток новых подписок, либо сокращение завершений. Выбор между этими направлениями требует дополнительной оценки эффективности и стоимости привлечения и удержания